# データを見る：日本語表示・34表

まず「基本台帳」から選んでください。「比較・診断」は別の仮定や取得版を比較する派生表です。
**通常画面の実績システムプライスはspot価格**です。感応度基準・原列・監査列は詳細で保持します。

このノートブックは読み取り専用です。元ファイルを改名・再計算せず、英語標準名の仮想表を日本語で表示します。
列対応表で「日本語名・英語名・単位・実際の型・元列・定義」を確認できます。

時刻は日本時間、コマ1＝00:00–00:30、コマ48＝23:30–24:00。
需要は元頻度のまま。原本一覧は [原本取得完了報告](../reports/issue_4_closeout.md)、全体の入口は [README](../README.md)。

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "metadata/view_table_registry.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "metadata/view_table_registry.json").exists():
    raise RuntimeError("共有repo直下またはnotebooksから開いてください")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import csv
import json

from IPython.display import display

from src.data_browser import japanese_display, read_page

TABLES = json.loads((ROOT / "metadata/view_table_registry.json").read_text())["tables"]
with (ROOT / "metadata/view_column_dictionary.csv").open() as f:
    COLUMNS = list(csv.DictReader(f))
print(f"閲覧対象 {len(TABLES)}表。ファイルは共有repoのdata/にあります。")

In [ ]:
def show_table(table_id="t01", *, detail=False, offset=0, limit=10, filters=None):
    """返り値は英語標準名のArrow表。画面は日本語。絞り込みは元値に対して行う。"""
    table = next(t for t in TABLES if t["table_id"] == table_id)
    columns = [r for r in COLUMNS if r["table_id"] == table_id]
    page, qa = read_page(
        ROOT, table, columns, detail=detail, offset=offset, limit=limit, filters=filters
    )
    print(table["title_ja"])
    print(f"全{table['rows']:,}行／条件一致{qa['matched_rows']:,}行／表示{len(page)}行")
    print(table["source_path"])
    display(japanese_display(page, columns))
    return page


def show_dictionary(table_id="t01"):
    import pandas as pd

    columns = [r for r in COLUMNS if r["table_id"] == table_id]
    view = pd.DataFrame(columns)[
        [
            "japanese_name",
            "standard_english_name",
            "unit",
            "arrow_type",
            "column_group",
            "source_column",
            "definition_ja",
        ]
    ]
    view.columns = ["日本語名", "英語標準列名", "単位", "型", "列の役割", "元列名", "定義"]
    display(view)


preview = show_table()

## 表の選択

以下のメニューで切り替えられます。日付・地域の絞り込みは、下の関数でも指定できます。
詳細表示は原列・監査列を含みます。通常表示は主要列の抜粋なので、全列は対応表で確認してください。

`show_table("t13", filters={"area": "北海道"})` のような条件は、その表に列がある場合だけ使えます。
日付列には `datetime.date(2025, 4, 1)` を指定します。

停止記録の2取得版・復旧予定日の2端点は比較条件です。**4表を独立した市場観測として足しません。**

In [ ]:
import ipywidgets as widgets

category = widgets.Dropdown(options=["基本台帳", "比較・診断（詳細）"], description="種類")
selector = widgets.Dropdown(layout=widgets.Layout(width="95%"), description="表")
detail = widgets.Checkbox(value=False, description="詳細・監査も表示")
button = widgets.Button(description="表示する")
dictionary_button = widgets.Button(description="列名・型・定義を開く")
output = widgets.Output()


def refresh_options(*_):
    selector.options = [
        (f"{t['table_id']}：{t['title_ja']}", t["table_id"])
        for t in TABLES
        if t["category"] == category.value
    ]


def render(_):
    with output:
        output.clear_output(wait=True)
        show_table(selector.value, detail=detail.value)


def render_dictionary(_):
    with output:
        output.clear_output(wait=True)
        show_dictionary(selector.value)


category.observe(refresh_options, names="value")
button.on_click(render)
dictionary_button.on_click(render_dictionary)
refresh_options()
display(category, selector, detail, widgets.HBox([button, dictionary_button]), output)

## よくある疑問

- 「案A」は原本ではなく、公表停止記録を発電実績に併記した**診断用の派生表**です。
- 7,828／58,852は元停止CSVの行数です。どちらの市場結合表も157,680行です。
- 復旧予定日には時刻がないため、その日の始めまで／終わりまでを比較します。実際の復旧時刻ではありません。
- 台帳の事業者名は所有・売電権限・契約供給力の帰属を確定するものではありません。
- 通常画面で感応度基準を出さなくても、公表差分の基準をspot価格で上書きした意味ではありません。

具体例は [データの読み方](../docs/data_reading_faq.md) を参照してください。